# Subject Dependent Experiments
model trained on one individual subject's samples, shuffled and divided into an 80-20 train-test split

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from Dataloader import read_excel_eeg, read_excel_nirs
from Model_MI_PyTorch import ef_net
from Metrics import metrics

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, cohen_kappa_score
import os
import pandas as pd


class Trainer:
	def __init__(self, config):
		self.config = config
		self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
		self.sampling_points = config['sampling_points']
		self.flooding_level = config['flooding_level']

		self.model = ef_net().to(self.device)
		self.criterion = nn.CrossEntropyLoss()
		self.optimizer = torch.optim.AdamW(self.model.parameters())

		os.makedirs('Results', exist_ok=True)
	
	def train_epoch(self, epoch, train_loader):
		self.model.train()
		total_correct, total_loss = 0, 0

		for batch_eeg, batch_nirs, batch_labels in train_loader:
			batch_eeg = batch_eeg.to(self.device, dtype=torch.float32)
			batch_nirs = batch_nirs.to(self.device, dtype=torch.float32)
			batch_labels = batch_labels.to(self.device)
			
			outputs = self.model(batch_eeg, batch_nirs)
			loss = self.criterion(outputs, batch_labels.long())

			if epoch < 30:
				b = self.flooding_level[0]
			elif epoch < 50:
				b = self.flooding_level[1]
			else:
				b = self.flooding_level[2]

			loss = (loss - b).abs() + b
			self. optimizer.zero_grad()
			loss.backward()
			self.optimizer.step()
			
			preds = torch.argmax(outputs, dim=1)
			total_correct += (preds == batch_labels).sum().item()
			total_loss += loss.item()

		train_loss = total_loss / len(train_loader)
		train_acc = total_correct / len(train_loader.dataset)
		
		return train_loss, train_acc
	
	def evaluate_epoch(self, epoch, eval_loader):
		self.model.eval()
		total_loss, total_correct = 0, 0
		all_preds, all_labels = [], []
		
		with torch.no_grad():
			for eval_eeg, eval_nirs, eval_labels in eval_loader:
				eval_eeg = eval_eeg.to(self.device, dtype=torch.float32)
				eval_nirs = eval_nirs.to(self.device, dtype=torch.float32)
				eval_labels = eval_labels.to(self.device)
				
				outputs = self.model(eval_eeg, eval_nirs)
				loss = self.criterion(outputs, eval_labels.long())
				
				preds = torch.argmax(outputs, dim=1)
				total_correct += (preds == eval_labels).sum().item()
				total_loss += loss.item()
				
				all_preds.extend(preds.cpu().numpy())
				all_labels.extend(eval_labels.cpu().numpy())
		
		loss = total_loss / len(eval_loader)
		acc = total_correct / len(eval_loader.dataset)
		precision = precision_score(all_labels, all_preds, zero_division=0)
		recall = recall_score(all_labels, all_preds, zero_division=0)
		f1 = f1_score(all_labels, all_preds, zero_division=0)
		kappa = cohen_kappa_score(all_labels, all_preds)
		
		return loss, acc, precision, recall, f1, kappa

	def train_subject(self, subject, mode):
		seed= 42
		torch.manual_seed(seed)
		eeg, labels = read_excel_eeg(subject, mode)
		nirs = read_excel_nirs(subject, mode)

		# Standardize input signals
		scaler = StandardScaler()
		eeg = scaler.fit_transform(eeg.reshape(-1, eeg.shape[-1])).reshape(eeg.shape)
		nirs = scaler.fit_transform(nirs.reshape(-1, nirs.shape[-1])).reshape(nirs.shape)

		print(eeg.shape)
		print(nirs.shape)
		print(labels.shape)

		eeg = torch.tensor(eeg, dtype=torch.float32)
		nirs = torch.tensor(nirs, dtype=torch.float32)
		labels = torch.tensor(labels, dtype=torch.float32)
		eeg, nirs, labels = eeg.to(self.device), nirs.to(self.device), labels.to(self.device)

		train_eeg, test_eeg, train_labels, test_labels = train_test_split(eeg, labels, test_size=0.4, random_state=self.config['RANDOM_STATE'])
		train_nirs, test_nirs, _, _ = train_test_split(nirs, labels, test_size=0.4, random_state=self.config['RANDOM_STATE'])
		
		train_dataset = TensorDataset(train_eeg, train_nirs, train_labels)
		train_loader = DataLoader(train_dataset, batch_size=self.config['batch_size'], shuffle=True)
		
		eval_dataset = TensorDataset(test_eeg, test_nirs, test_labels)
		eval_loader = DataLoader(eval_dataset, batch_size=self.config['batch_size'], shuffle=False)
		
		acc_list, precision_list, recall_list, f1_list, kappa_list = [], [], [], [], []
		
		for epoch in range(self.config['num_epochs']):
			train_loss, train_acc = self.train_epoch(epoch, train_loader)
			eval_loss, eval_acc, precision, recall, f1, kappa = self.evaluate_epoch(epoch, eval_loader)
			
			acc_list.append(eval_acc)
			precision_list.append(precision)
			recall_list.append(recall)
			f1_list.append(f1)
			kappa_list.append(kappa)
			
			print(f"Subject {subject} | Epoch {epoch+1}/{self.config['num_epochs']} | "
				f"Train Loss: {train_loss:.2f} | Train Acc: {train_acc:.2f} | "
				f"Eval Loss: {eval_loss:.2f} | Eval Acc: {eval_acc:.2f}")

		metrics(subject, config['log_name'], config['log_mode'], acc_list, precision_list, recall_list, f1_list, kappa_list)
		return

# Hyperparameters
config = {
	'depth': 4,
	'query_size': 64,
	'key_size': 64,
	'value_size': 64,
	'emb_size': 64,
	'decay': 0.99,
	'num_heads': 4,
	'expansion': 2,
	'conv_dropout': 0.3,
	'self_dropout': 0.3,
	'cross_dropout': 0.3,
	'cls_dropout': 0.5,
	'num_classes': 2,
	'batch_size': 32,
	'num_epochs': 70,
	'learning_rate': 1e-3,
	'sampling_points': 100,
	'flooding_level': [0.45, 0.40, 0.35],
	'baseline': False,
	'RANDOM_STATE': 38,
	'log_name': 68,
	'log_mode': 1,
}

# Initialize and run trainer
trainer = Trainer(config)
for subject in range(29):
	subject += 1
	print(f"\n=== Subject {subject} ===")
	results = trainer.train_subject(subject, mode=0) # 0 = MI, 1 = MA

/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(



=== Subject 1 ===
1 EEG torch.Size([60, 4000, 30, 1]) Labels torch.Size([60])
1 fNIRS torch.Size([60, 200, 36, 1]) Labels (60,)
(60, 4000, 30, 1)
(60, 200, 36, 1)
torch.Size([60])


/tmp/ipykernel_922431/2612874024.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels, dtype=torch.float32)


Subject 1 | Epoch 1/70 | Train Loss: 0.69 | Train Acc: 0.58 | Eval Loss: 0.70 | Eval Acc: 0.29
Subject 1 | Epoch 2/70 | Train Loss: 0.69 | Train Acc: 0.64 | Eval Loss: 0.70 | Eval Acc: 0.29
Subject 1 | Epoch 3/70 | Train Loss: 0.69 | Train Acc: 0.64 | Eval Loss: 0.70 | Eval Acc: 0.29
Subject 1 | Epoch 4/70 | Train Loss: 0.69 | Train Acc: 0.64 | Eval Loss: 0.70 | Eval Acc: 0.29
Subject 1 | Epoch 5/70 | Train Loss: 0.69 | Train Acc: 0.64 | Eval Loss: 0.70 | Eval Acc: 0.29
Subject 1 | Epoch 6/70 | Train Loss: 0.69 | Train Acc: 0.64 | Eval Loss: 0.71 | Eval Acc: 0.29
Subject 1 | Epoch 7/70 | Train Loss: 0.68 | Train Acc: 0.64 | Eval Loss: 0.71 | Eval Acc: 0.29
Subject 1 | Epoch 8/70 | Train Loss: 0.69 | Train Acc: 0.64 | Eval Loss: 0.71 | Eval Acc: 0.29
Subject 1 | Epoch 9/70 | Train Loss: 0.69 | Train Acc: 0.64 | Eval Loss: 0.71 | Eval Acc: 0.29
Subject 1 | Epoch 10/70 | Train Loss: 0.69 | Train Acc: 0.64 | Eval Loss: 0.71 | Eval Acc: 0.29
Subject 1 | Epoch 11/70 | Train Loss: 0.67 | Trai

/tmp/ipykernel_922431/2612874024.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels, dtype=torch.float32)


...

Subject 6 | Epoch 2/70 | Train Loss: 0.56 | Train Acc: 0.75 | Eval Loss: 0.76 | Eval Acc: 0.54
Subject 6 | Epoch 3/70 | Train Loss: 0.56 | Train Acc: 0.75 | Eval Loss: 0.81 | Eval Acc: 0.46
Subject 6 | Epoch 4/70 | Train Loss: 0.54 | Train Acc: 0.81 | Eval Loss: 0.86 | Eval Acc: 0.42
Subject 6 | Epoch 5/70 | Train Loss: 0.54 | Train Acc: 0.78 | Eval Loss: 0.85 | Eval Acc: 0.42
Subject 6 | Epoch 6/70 | Train Loss: 0.53 | Train Acc: 0.78 | Eval Loss: 0.84 | Eval Acc: 0.46
Subject 6 | Epoch 7/70 | Train Loss: 0.50 | Train Acc: 0.83 | Eval Loss: 0.81 | Eval Acc: 0.42
Subject 6 | Epoch 8/70 | Train Loss: 0.55 | Train Acc: 0.89 | Eval Loss: 0.78 | Eval Acc: 0.54
Subject 6 | Epoch 9/70 | Train Loss: 0.49 | Train Acc: 0.89 | Eval Loss: 0.79 | Eval Acc: 0.46
Subject 6 | Epoch 10/70 | Train Loss: 0.56 | Train Acc: 0.89 | Eval Loss: 0.82 | Eval Acc: 0.46
Subject 6 | Epoch 11/70 | Train Loss: 0.51 | Train Acc: 0.83 | Eval Loss: 0.87 | Eval Acc: 0.38
Subject 6 | Epoch 12/70 | Train Loss: 0.51 | Tra

/tmp/ipykernel_922431/2612874024.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels, dtype=torch.float32)


Subject 7 | Epoch 1/70 | Train Loss: 0.54 | Train Acc: 0.81 | Eval Loss: 0.90 | Eval Acc: 0.38
Subject 7 | Epoch 2/70 | Train Loss: 0.52 | Train Acc: 0.81 | Eval Loss: 0.86 | Eval Acc: 0.46
Subject 7 | Epoch 3/70 | Train Loss: 0.47 | Train Acc: 0.83 | Eval Loss: 0.81 | Eval Acc: 0.46
Subject 7 | Epoch 4/70 | Train Loss: 0.55 | Train Acc: 0.78 | Eval Loss: 0.72 | Eval Acc: 0.58
Subject 7 | Epoch 5/70 | Train Loss: 0.55 | Train Acc: 0.78 | Eval Loss: 0.69 | Eval Acc: 0.62
Subject 7 | Epoch 6/70 | Train Loss: 0.51 | Train Acc: 0.89 | Eval Loss: 0.67 | Eval Acc: 0.67
Subject 7 | Epoch 7/70 | Train Loss: 0.51 | Train Acc: 0.89 | Eval Loss: 0.70 | Eval Acc: 0.62
Subject 7 | Epoch 8/70 | Train Loss: 0.53 | Train Acc: 0.83 | Eval Loss: 0.75 | Eval Acc: 0.54
Subject 7 | Epoch 9/70 | Train Loss: 0.52 | Train Acc: 0.81 | Eval Loss: 0.81 | Eval Acc: 0.46
Subject 7 | Epoch 10/70 | Train Loss: 0.52 | Train Acc: 0.83 | Eval Loss: 0.84 | Eval Acc: 0.42
Subject 7 | Epoch 11/70 | Train Loss: 0.48 | Trai

/tmp/ipykernel_922431/2612874024.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels, dtype=torch.float32)


Subject 8 | Epoch 1/70 | Train Loss: 0.51 | Train Acc: 0.89 | Eval Loss: 0.88 | Eval Acc: 0.42
Subject 8 | Epoch 2/70 | Train Loss: 0.51 | Train Acc: 0.86 | Eval Loss: 0.86 | Eval Acc: 0.42
Subject 8 | Epoch 3/70 | Train Loss: 0.52 | Train Acc: 0.83 | Eval Loss: 0.85 | Eval Acc: 0.42
Subject 8 | Epoch 4/70 | Train Loss: 0.50 | Train Acc: 0.86 | Eval Loss: 0.80 | Eval Acc: 0.54
Subject 8 | Epoch 5/70 | Train Loss: 0.50 | Train Acc: 0.86 | Eval Loss: 0.80 | Eval Acc: 0.50
Subject 8 | Epoch 6/70 | Train Loss: 0.51 | Train Acc: 0.83 | Eval Loss: 0.80 | Eval Acc: 0.50
Subject 8 | Epoch 7/70 | Train Loss: 0.50 | Train Acc: 0.83 | Eval Loss: 0.75 | Eval Acc: 0.54
Subject 8 | Epoch 8/70 | Train Loss: 0.49 | Train Acc: 0.83 | Eval Loss: 0.75 | Eval Acc: 0.54
Subject 8 | Epoch 9/70 | Train Loss: 0.52 | Train Acc: 0.94 | Eval Loss: 0.74 | Eval Acc: 0.54
Subject 8 | Epoch 10/70 | Train Loss: 0.54 | Train Acc: 0.92 | Eval Loss: 0.73 | Eval Acc: 0.58
Subject 8 | Epoch 11/70 | Train Loss: 0.45 | Trai